In [69]:
import pandas as pd 

df_combined = pd.read_csv("combined_yearly.csv")
print(df_combined.columns)
# print(df_combined.head(2))
print(df_combined.shape)
# print(df_combined.describe())
print(df_combined.info())


Index(['country', 'commodity', 'year', 'cost_healthy_diet_ppp_usd',
       'annual_cost_healthy_diet_usd', 'cost_vegetables_ppp_usd',
       'cost_fruits_ppp_usd', 'total_food_components_cost', 'continent'],
      dtype='str')
(1379, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1379 entries, 0 to 1378
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       1379 non-null   str    
 1   commodity                     0 non-null      float64
 2   year                          1379 non-null   float64
 3   cost_healthy_diet_ppp_usd     1379 non-null   float64
 4   annual_cost_healthy_diet_usd  1379 non-null   float64
 5   cost_vegetables_ppp_usd       166 non-null    float64
 6   cost_fruits_ppp_usd           166 non-null    float64
 7   total_food_components_cost    166 non-null    float64
 8   continent                     1209 non-null   str    
dtypes: float64(7),

In [70]:
df_combined['year'] = df_combined['year'].astype(int, errors='ignore')
df_combined.drop('commodity', axis=1, inplace=True, errors='ignore')
# print(df_combined.shape)
# print(df_combined.info())

print(df.memory_usage());
print(df.memory_usage(deep=True));



Index                             132
id                              11032
country                         11032
year                            11032
cost_healthy_diet_ppp_usd       11032
annual_cost_healthy_diet_usd    11032
cost_vegetables_ppp_usd         11032
cost_fruits_ppp_usd             11032
total_food_components_cost      11032
continent                       11032
cpi                             11032
dtype: int64
Index                             132
id                              11032
country                         81933
year                            11032
cost_healthy_diet_ppp_usd       11032
annual_cost_healthy_diet_usd    11032
cost_vegetables_ppp_usd         11032
cost_fruits_ppp_usd             11032
total_food_components_cost      11032
continent                       73263
cpi                             11032
dtype: int64


In [61]:
df_cpi_data = pd.read_csv("flat-ui__data-Fri Apr 04 2025.csv")
df_cpi_data.drop(columns=['Country Code'], inplace=True)
print(df_cpi_data.columns)

df_cpi_data.rename(columns={'Country': 'country'}, inplace=True)
df_cpi_data.rename(columns={'Year': 'year'}, inplace=True)
df_cpi_data.rename(columns={'CPI': 'cpi'}, inplace=True)
print(df_cpi_data.columns)

df_cpi_data['year'] = df_cpi_data['year'].astype(int)

# Step 4 - Merge with combined_yearly (left keeps ALL rows from combined_yearly)
new_df = df_combined.merge(df_cpi_data, on=['country', 'year'], how='left')

print(new_df.head())
print('\nShape:', new_df.shape)
print('Missing values:\n', new_df.isna().sum())



Index(['Country', 'Year', 'CPI'], dtype='str')
Index(['country', 'year', 'cpi'], dtype='str')
   country  year  cost_healthy_diet_ppp_usd  annual_cost_healthy_diet_usd  \
0  Albania  2017                       3.04                       1109.60   
1  Albania  2018                       3.13                       1142.45   
2  Albania  2019                       3.32                       1211.80   
3  Albania  2020                       3.40                       1241.00   
4  Albania  2021                       3.49                       1273.85   

   cost_vegetables_ppp_usd  cost_fruits_ppp_usd  total_food_components_cost  \
0                      NaN                  NaN                         NaN   
1                      NaN                  NaN                         NaN   
2                      NaN                  NaN                         NaN   
3                      NaN                  NaN                         NaN   
4                      0.6                 0.77 

In [62]:
# df = pd.read_csv("consumer_price_index.csv")
# print(df.info())
# print(df.shape)
# # print(df.describe)
# # print(df.head())
# print(df.columns)
# print(df.info())

# df_cpi = pd.read_csv('consumer_price_index.csv', low_memory=False)

# # Step 1 - Filter to annual rows only
# df_cpi_annual = df_cpi[df_cpi['FREQ'] == 'A'].copy()

# # Step 2 - Keep only country + relevant years
# relevant_years = ['2017', '2018', '2019', '2020', '2021', '2022', '2023']
# df_cpi_clean = df_cpi_annual[['Reference area'] + relevant_years].copy()
# df_cpi_clean.rename(columns={'Reference area': 'country'}, inplace=True)
# print(df_cpi_clean.columns)

# # Step 3 - Melt to long format
# df_cpi_long = df_cpi_clean.melt(
#     id_vars='country',
#     value_vars=relevant_years,
#     var_name='year',
#     value_name='cpi'
# )
# df_cpi_long['year'] = df_cpi_long['year'].astype(int)

# # Step 4 - Merge with combined_yearly (left keeps ALL rows from combined_yearly)
# df_merged = df_combined.merge(df_cpi_long, on=['country', 'year'], how='left')

# print(df_merged.head())
# print('\nShape:', df_merged.shape)


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('foodprices_data.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS FPtable")

cursor.execute("""
    CREATE TABLE FPtable (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        country TEXT,
        year SMALLINT,
        cost_healthy_diet_ppp_usd FLOAT,    
        annual_cost_healthy_diet_usd FLOAT,
        cost_vegetables_ppp_usd FLOAT,
        cost_fruits_ppp_usd FLOAT,
        total_food_components_cost FLOAT,
        continent TEXT,
        cpi FLOAT
    )
""")

# Verify schema BEFORE inserting
cursor.execute("PRAGMA table_info(FPtable)")
schema = cursor.fetchall()
print("Schema:", schema)
print("Number of columns:", len(schema)) 

# Prepare rows
columns = [
    'country',
    'year', 
    'cost_healthy_diet_ppp_usd',
    'annual_cost_healthy_diet_usd',
    'cost_vegetables_ppp_usd',
    'cost_fruits_ppp_usd',
    'total_food_components_cost',
    'continent',
    'cpi'
]
rows = list(new_df[columns].itertuples(index=False, name=None))

# Insert data
cursor.executemany("""
    INSERT INTO FPtable (country, year, cost_healthy_diet_ppp_usd, annual_cost_healthy_diet_usd,
                         cost_vegetables_ppp_usd, cost_fruits_ppp_usd, total_food_components_cost, 
                         continent, cpi)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", rows)

conn.commit()
print(f"Inserted {len(rows)} rows!")

# Verify output
cursor.execute("SELECT * FROM FPtable LIMIT 5")
for row in cursor.fetchall():
    print(row)

conn.close()

Schema: [(0, 'id', 'INTEGER', 0, None, 1), (1, 'country', 'TEXT', 0, None, 0), (2, 'year', 'INTEGER', 0, None, 0), (3, 'cost_healthy_diet_ppp_usd', 'FLOAT', 0, None, 0), (4, 'annual_cost_healthy_diet_usd', 'FLOAT', 0, None, 0), (5, 'cost_vegetables_ppp_usd', 'FLOAT', 0, None, 0), (6, 'cost_fruits_ppp_usd', 'FLOAT', 0, None, 0), (7, 'total_food_components_cost', 'FLOAT', 0, None, 0), (8, 'continent', 'TEXT', 0, None, 0), (9, 'cpi', 'FLOAT', 0, None, 0)]
Number of columns: 10
Inserted 1379 rows!
(1, 'Albania', 2017, 3.04, 1109.6, None, None, None, 'Europe', 1.98666133171199)
(2, 'Albania', 2018, 3.13, 1142.45, None, None, None, 'Europe', 2.02805963071136)
(3, 'Albania', 2019, 3.32, 1211.8, None, None, None, 'Europe', 1.41109078954246)
(4, 'Albania', 2020, 3.4, 1241.0, None, None, None, 'Europe', 1.62088661717011)
(5, 'Albania', 2021, 3.49, 1273.85, 0.6, 0.77, 1.37, 'Europe', 2.04147163139549)


In [64]:
# import sqlite3
# import pandas as pd

# conn = sqlite3.connect('foodprices_data.db')
# cursor = conn.cursor()

# cursor.execute("DROP TABLE IF EXISTS FPtable")

# cursor.execute("""
#     CREATE TABLE FPtable (
#         id INTEGER PRIMARY KEY AUTOINCREMENT,
#         country TEXT,
#         year INTEGER,
#         cost_healthy_diet_ppp_usd FLOAT,    
#         annual_cost_healthy_diet_usd FLOAT,
#         cost_vegetables_ppp_usd FLOAT,
#         cost_fruits_ppp_usd FLOAT,
#         total_food_components_cost FLOAT,
#         continent TEXT,
#         cpi FLOAT
#     )
# """)

# # Verify schema BEFORE inserting
# cursor.execute("PRAGMA table_info(FPtable)")
# schema = cursor.fetchall()
# print("Schema:", schema)
# print("Number of columns:", len(schema)) 

# # Prepare rows
# columns = [
#     'country',
#     'year', 
#     'cost_healthy_diet_ppp_usd',
#     'annual_cost_healthy_diet_usd',
#     'cost_vegetables_ppp_usd',
#     'cost_fruits_ppp_usd',
#     'total_food_components_cost',
#     'continent',
#     'cpi'
# ]
# rows = list(df_merged[columns].itertuples(index=False, name=None))

# # Insert data
# cursor.executemany("""
#     INSERT INTO FPtable (country, year, cost_healthy_diet_ppp_usd, annual_cost_healthy_diet_usd,
#                          cost_vegetables_ppp_usd, cost_fruits_ppp_usd, total_food_components_cost, 
#                          continent, cpi)
#     VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
# """, rows)

# conn.commit()
# print(f"Inserted {len(rows)} rows!")

# # Verify output
# cursor.execute("SELECT * FROM FPtable LIMIT 5")
# for row in cursor.fetchall():
#     print(row)

# conn.close()

In [65]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('foodprices_data.db')

# Read entire table into a DataFrame
df = pd.read_sql_query("SELECT * FROM FPtable", conn)

# Save to CSV
df.to_csv('FP_output.csv', index=False)

conn.close()
print(f"Exported {len(df)} rows to FP_output.csv")

Exported 1379 rows to FP_output.csv
